In [1]:
# Sentiment Analysis for Stock and Twitter Data
# This notebook performs sentiment analysis on tweets and correlates with stock price movements

import pandas as pd
from datetime import timedelta
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import os

# Initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

In [2]:
# Variable to do for the given ticker
ticker = "AMZN"

In [3]:
# Load the data files
print("Loading stock market data...")
stock_data = pd.read_csv(f'../data/{ticker}/{ticker}_market_clean.csv')
print(f"Date range: {stock_data['date'].min()} to {stock_data['date'].max()}")

print("\nLoading tweet data...")
tweet_data = pd.read_csv(f'../data/{ticker}/{ticker}_tweets_clean.csv')
print(f"Date range: {tweet_data['created_at'].min()} to {tweet_data['created_at'].max()}")

Loading stock market data...
Date range: 2020-10-01 to 2025-09-30

Loading tweet data...
Date range: 2020-10-01 to 2025-09-30


In [4]:
# Data preprocessing and exploration
print("Data preprocessing...")

# Convert date columns to datetime
stock_data['date'] = pd.to_datetime(stock_data['date'])
tweet_data['created_at'] = pd.to_datetime(tweet_data['created_at'])

# Extract date from tweet timestamps (keep only the date part)
tweet_data['date'] = tweet_data['created_at'].dt.date

# Check unique dates in each dataset
print(f"\nUnique dates in stock data: {len(stock_data['date'].dt.date.unique())}")
print(f"Unique dates in tweet data: {len(tweet_data['date'].unique())}")

Data preprocessing...

Unique dates in stock data: 1255
Unique dates in tweet data: 1728


In [5]:
# VADER sentiment analysis
def analyze_sentiment(text):
    """
    Analyze sentiment using VADER (Valence Aware Dictionary and sEntiment Reasoner)
    Returns polarity score (-1 to 1, where -1 is very negative, 1 is very positive)
    VADER is specifically designed for social media text and handles emojis, slang, etc.
    """
    try:
        # Get sentiment scores from VADER
        scores = analyzer.polarity_scores(str(text))
        
        # Return the compound score which is the overall sentiment
        # Compound score ranges from -1 (most negative) to +1 (most positive)
        return scores['compound']
    except:
        return 0.0  # Return neutral sentiment for problematic text

# Apply sentiment analysis to tweets
print("Performing sentiment analysis on tweets...")
tweet_data['sentiment'] = tweet_data['text'].apply(analyze_sentiment)

# Show distribution of sentiment scores
print(f"\nSentiment distribution:")
print(tweet_data['sentiment'].describe())


Performing sentiment analysis on tweets...

Sentiment distribution:
count    36833.000000
mean         0.215175
std          0.410511
min         -0.983500
25%          0.000000
50%          0.177900
75%          0.542300
max          0.999700
Name: sentiment, dtype: float64


In [6]:
# Group tweets by date and calculate average sentiment
print("Grouping tweets by date and calculating average sentiment...")

# Group by date and calculate average sentiment
daily_sentiment = tweet_data.groupby('date')['sentiment'].agg(['mean', 'count']).reset_index()
daily_sentiment.columns = ['date', 'average_sentiment', 'tweet_count']

# Show first few rows
print("\nDaily sentiment preview:")
print(daily_sentiment.head())

# Show tweet count distribution
print(f"\nTweet count per day statistics:")
print(daily_sentiment['tweet_count'].describe())


Grouping tweets by date and calculating average sentiment...

Daily sentiment preview:
         date  average_sentiment  tweet_count
0  2020-10-01           0.285836           25
1  2020-10-02           0.309668           25
2  2020-10-03           0.224064           25
3  2020-10-04           0.311236           25
4  2020-10-05           0.294772           25

Tweet count per day statistics:
count    1728.000000
mean       21.315394
std         7.650861
min         1.000000
25%        25.000000
50%        25.000000
75%        25.000000
max        25.000000
Name: tweet_count, dtype: float64


In [7]:
# Handle weekend grouping - group Saturday and Sunday with the previous Friday

# Convert date to datetime for easier manipulation
daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])

# Create a mapping for weekend grouping
def get_grouping_date(date):
    """
    For weekends (Saturday=5, Sunday=6), group with the previous Friday
    For weekdays, keep the original date
    """
    if date.weekday() == 5:  # Saturday
        return date - timedelta(days=1)  # Group with Friday
    elif date.weekday() == 6:  # Sunday
        return date - timedelta(days=2)  # Group with Friday
    else:
        return date  # Keep original date for weekdays

# Apply weekend grouping
daily_sentiment['grouping_date'] = daily_sentiment['date'].apply(get_grouping_date)

# Group by the new grouping date and recalculate sentiment
grouped_sentiment = daily_sentiment.groupby('grouping_date').agg({
    'average_sentiment': 'mean',  # Average of averages
    'tweet_count': 'sum'  # Sum tweet counts
}).reset_index()

grouped_sentiment.columns = ['date', 'average_sentiment', 'tweet_count']

print(f"After weekend grouping:")
print(f"Grouped sentiment data shape: {grouped_sentiment.shape}")

# Show some examples of grouped data
print(f"\nGrouped sentiment preview:")
print(grouped_sentiment.head(5))


After weekend grouping:
Grouped sentiment data shape: (1246, 3)

Grouped sentiment preview:
        date  average_sentiment  tweet_count
0 2020-10-01           0.285836           25
1 2020-10-02           0.281656           75
2 2020-10-05           0.294772           25
3 2020-10-06           0.144200           25
4 2020-10-07           0.317184           25


In [8]:
# Calculate price differences (next day open - current day close)
print("Calculating price differences...")

# Sort stock data by date to ensure proper order
stock_data_sorted = stock_data.sort_values('date').reset_index(drop=True)

# Calculate the difference between next day's open and current day's close
stock_data_sorted['next_day_open'] = stock_data_sorted['open'].shift(-1)
stock_data_sorted['price_difference'] = stock_data_sorted['next_day_open'] - stock_data_sorted['close']

# Remove the last row since it won't have a next day
stock_data_sorted = stock_data_sorted[:-1]

print(f"Stock data with price differences shape: {stock_data_sorted.shape}")
print(f"Price difference range: {stock_data_sorted['price_difference'].min():.3f} to {stock_data_sorted['price_difference'].max():.3f}")
print(f"Mean price difference: {stock_data_sorted['price_difference'].mean():.3f}")

# Show first few rows with price differences
print(f"\nStock data with price differences preview:")
print(stock_data_sorted[['date', 'open', 'close', 'next_day_open', 'price_difference']].head(10))


Calculating price differences...
Stock data with price differences shape: (1254, 5)
Price difference range: -17.320 to 17.650
Mean price difference: 0.091

Stock data with price differences preview:
        date        open       close  next_day_open  price_difference
0 2020-10-01  160.399994  161.063004     157.681503         -3.381500
1 2020-10-02  157.681503  156.250000     157.292007          1.042007
2 2020-10-05  157.292007  159.960007     158.250000         -1.710007
3 2020-10-06  158.250000  154.998001     156.750000          1.751999
4 2020-10-07  156.750000  159.784500     161.249496          1.464996
5 2020-10-08  161.249496  159.527496     160.500000          0.972504
6 2020-10-09  160.500000  164.332504     167.496994          3.164490
7 2020-10-12  167.496994  172.146500     173.399506          1.253006
8 2020-10-13  173.399506  172.181503     172.350006          0.168503
9 2020-10-14  172.350006  168.185501     164.600494         -3.585007


In [9]:
# Merge sentiment data with stock data
print("Merging sentiment and stock data...")

# Convert date columns to the same format for merging
grouped_sentiment['date'] = pd.to_datetime(grouped_sentiment['date']).dt.date
stock_data_sorted['date'] = stock_data_sorted['date'].dt.date

# Merge the datasets
final_dataset = pd.merge(
    grouped_sentiment, 
    stock_data_sorted[['date', 'price_difference']], 
    on='date', 
    how='inner'  # Only keep dates that exist in both datasets
)

# Show the final dataset structure
print(f"\nFinal dataset preview:")
print(final_dataset.head(10))

# Show statistics for the final dataset
print(f"\nFinal dataset statistics:")
print(final_dataset.describe())


Merging sentiment and stock data...

Final dataset preview:
         date  average_sentiment  tweet_count  price_difference
0  2020-10-01           0.285836           25         -3.381500
1  2020-10-02           0.281656           75          1.042007
2  2020-10-05           0.294772           25         -1.710007
3  2020-10-06           0.144200           25          1.751999
4  2020-10-07           0.317184           25          1.464996
5  2020-10-08           0.133632           25          0.972504
6  2020-10-09           0.218413           75          3.164490
7  2020-10-12           0.354020           25          1.253006
8  2020-10-13           0.160952           25          0.168503
9  2020-10-14           0.259704           25         -3.585007

Final dataset statistics:
       average_sentiment  tweet_count  price_difference
count        1201.000000  1201.000000       1201.000000
mean            0.208697    29.706078          0.080903
std             0.136534    19.871698    

In [10]:
# Update the merge to include open and close prices with correct alignment
print("Updating merge to include open and close prices with correct alignment...")

# Create properly aligned stock data:
# - close: current day's close price
# - next_day_open: next day's open price (for price_difference calculation)
# - We need to shift the open prices to represent next day's open

# Create a copy of stock data with shifted open prices
stock_aligned = stock_data_sorted.copy()
stock_aligned['next_day_open'] = stock_data_sorted['open'].shift(-1)

# Re-merge the datasets with properly aligned prices
final_dataset = pd.merge(
    grouped_sentiment, 
    stock_aligned[['date', 'close', 'next_day_open', 'price_difference']], 
    on='date', 
    how='inner'  # Only keep dates that exist in both datasets
)

# Rename columns for clarity
final_dataset = final_dataset.rename(columns={
    'close': 'close',
    'next_day_open': 'open'  # This is actually the next day's open
})

print(f"Updated dataset shape: {final_dataset.shape}")
print(f"Updated dataset columns: {list(final_dataset.columns)}")
print(f"\nSample of aligned data:")
print(final_dataset[['date', 'open', 'close', 'price_difference']].head())


Updating merge to include open and close prices with correct alignment...
Updated dataset shape: (1201, 6)
Updated dataset columns: ['date', 'average_sentiment', 'tweet_count', 'close', 'open', 'price_difference']

Sample of aligned data:
         date        open       close  price_difference
0  2020-10-01  157.681503  161.063004         -3.381500
1  2020-10-02  157.292007  156.250000          1.042007
2  2020-10-05  158.250000  159.960007         -1.710007
3  2020-10-06  156.750000  154.998001          1.751999
4  2020-10-07  161.249496  159.784500          1.464996


In [11]:
# Create the final dataset with the required columns
print("Creating final sentiment dataset...")

# Select only the required columns: date, average_sentiment, price_difference
sentiment_final = final_dataset[['date', 'average_sentiment', 'price_difference']].copy()

# Show the final dataset
print(f"\nFinal sentiment dataset preview:")
print(sentiment_final.head(10))

# Show summary statistics
print(f"\nSummary statistics:")
print(sentiment_final.describe())


Creating final sentiment dataset...

Final sentiment dataset preview:
         date  average_sentiment  price_difference
0  2020-10-01           0.285836         -3.381500
1  2020-10-02           0.281656          1.042007
2  2020-10-05           0.294772         -1.710007
3  2020-10-06           0.144200          1.751999
4  2020-10-07           0.317184          1.464996
5  2020-10-08           0.133632          0.972504
6  2020-10-09           0.218413          3.164490
7  2020-10-12           0.354020          1.253006
8  2020-10-13           0.160952          0.168503
9  2020-10-14           0.259704         -3.585007

Summary statistics:
       average_sentiment  price_difference
count        1201.000000       1201.000000
mean            0.208697          0.080903
std             0.136534          2.336336
min            -0.670500        -17.320007
25%             0.143052         -0.720001
50%             0.209660          0.119995
75%             0.279816          1.000000
max 

In [12]:
# Update the final dataset to include open and close prices
print("Updating final dataset to include open and close prices...")

# Select the required columns: date, average_sentiment, open, close, price_difference
sentiment_final = final_dataset[['date', 'average_sentiment', 'open', 'close', 'price_difference']].copy()

print(f"Updated final dataset shape: {sentiment_final.shape}")
print(f"Updated final dataset columns: {list(sentiment_final.columns)}")


Updating final dataset to include open and close prices...
Updated final dataset shape: (1201, 5)
Updated final dataset columns: ['date', 'average_sentiment', 'open', 'close', 'price_difference']


In [13]:
# Save the final dataset
print("Saving the final sentiment dataset...")

# Save to the specified location
output_path = f'../data/{ticker}/{ticker}_sentiment.csv'
sentiment_final.to_csv(output_path, index=False)

print(f"Dataset saved to: {output_path}")

# Verify the saved file
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path)
    print(f"File saved successfully! Size: {file_size} bytes")
    
    # Read back a few rows to verify
    verification = pd.read_csv(output_path)
    print(f"\nVerification - first 5 rows of saved file:")
    print(verification.head())
    print(f"\nColumns in saved file: {list(verification.columns)}")
else:
    print("Error: File was not saved successfully!")


Saving the final sentiment dataset...
Dataset saved to: ../data/AMZN/AMZN_sentiment.csv
File saved successfully! Size: 94406 bytes

Verification - first 5 rows of saved file:
         date  average_sentiment        open       close  price_difference
0  2020-10-01           0.285836  157.681503  161.063004         -3.381500
1  2020-10-02           0.281656  157.292007  156.250000          1.042007
2  2020-10-05           0.294772  158.250000  159.960007         -1.710007
3  2020-10-06           0.144200  156.750000  154.998001          1.751999
4  2020-10-07           0.317184  161.249496  159.784500          1.464996

Columns in saved file: ['date', 'average_sentiment', 'open', 'close', 'price_difference']
